<h2>Description</h2>
- This is another prof of concept NB, which has been already used to summarize a single PDF file.<br>
- It works well, i.e. it solves the problem found in "testLocalLLM" NB.

In [2]:
# pip install tiktoken openai pypdf
import tiktoken
from openai import OpenAI
from pypdf import PdfReader

In [3]:
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")
MODEL = "qwen3-8b-instruct"  # loaded in LM Studio with >=8k context
enc = tiktoken.get_encoding("cl100k_base")

            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Summarize this section into 3–5 bullet points:\n\n{text}"}

In [13]:
def tokenize(txt): return enc.encode(txt)
def detokenize(ids): return enc.decode(ids)

def chunk_by_tokens(text, max_tokens=2500, overlap=200):
    ids = tokenize(text)
    i = 0
    while i < len(ids):
        j = min(i + max_tokens, len(ids))
        yield detokenize(ids[i:j])
        i = j - overlap if j - overlap > i else j

SYSTEM = "You write formal academic abstracts. Be concise and factual."

def summarize(text):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": f"Write a formal academic abstract (~200 words) as one paragraph. Cover motivation, method, and key results.\n\n{text}"}
        ],
        temperature=0.3,
        # keep some room for output; adjust if needed
        max_tokens=400
    )
    return resp.choices[0].message.content.strip()

In [14]:
# 1) read PDF
def read_pdf(path):
    r = PdfReader(path)
    return "\n".join((p.extract_text() or "") for p in r.pages)

filePath ="pdf/test.pdf"
full_text = read_pdf(filePath)

In [15]:
full_text

"Convolutional Neural Networks for Sentence Classiﬁcation\nYoon Kim\nNew York University\nyhk255@nyu.edu\nAbstract\nWe report on a series of experiments with\nconvolutional neural networks (CNN)\ntrained on top of pre-trained word vec-\ntors for sentence-level classiﬁcation tasks.\nWe show that a simple CNN with lit-\ntle hyperparameter tuning and static vec-\ntors achieves excellent results on multi-\nple benchmarks. Learning task-speciﬁc\nvectors through ﬁne-tuning offers further\ngains in performance. We additionally\npropose a simple modiﬁcation to the ar-\nchitecture to allow for the use of both\ntask-speciﬁc and static vectors. The CNN\nmodels discussed herein improve upon the\nstate of the art on 4 out of 7 tasks, which\ninclude sentiment analysis and question\nclassiﬁcation.\n1 Introduction\nDeep learning models have achieved remarkable\nresults in computer vision (Krizhevsky et al.,\n2012) and speech recognition (Graves et al., 2013)\nin recent years. Within natural language p

In [16]:
# 2) first pass: per-chunk summaries
first_pass = [summarize(chunk) for chunk in chunk_by_tokens(full_text, max_tokens=2500, overlap=200)]

In [17]:
# 3) second pass: reduce in batches to avoid overflow
def reduce_summaries(items, batch_size=6):
    out = []
    for i in range(0, len(items), batch_size):
        batch = "\n\n".join(items[i:i+batch_size])
        out.append(summarize(f"Combine and condense into 5 bullets:\n\n{batch}"))
    return out

In [18]:
level2 = reduce_summaries(first_pass, batch_size=6)

In [19]:
final_summary = summarize(f"Produce a ~200-word abstract covering motivation, method, and key results:\n\n" + "\n\n".join(level2))
print("FINAL SUMMARY:\n", final_summary)

FINAL SUMMARY:
 <think>
</think>

This study investigates the effectiveness of convolutional neural networks (CNNs) in sentence classification tasks by leveraging pre-trained word vectors, such as Word2Vec, to capture semantic features efficiently. Motivated by the limitations of traditional NLP models in capturing contextual dependencies and overfitting during training, we explore three variants of CNN architectures: static (fixed), non-static (fine-tuned), and multichannel (combined) representations. To ensure robustness, hyperparameters were systematically tuned via grid search on the SST-2 development set using stochastic gradient descent with Adadelta optimization. Experimental results demonstrate that static vectors achieve strong performance without fine-tuning, underscoring their utility in transfer learning scenarios. Non-static vectors further enhance accuracy by allowing word representations to adapt during task-specific training, highlighting the importance of contextual ad

In [12]:
# 3) Final abstract
final_summary = summarize(
    "Write a formal academic abstract (~200 words) as one paragraph. "
    "Cover motivation, method, and key results clearly.\n\n"
    + "\n\n".join(level2)
)

print("FINAL ABSTRACT:\n", final_summary)

FINAL ABSTRACT:
 <think>
</think>

**Abstract**  
This study investigates the application of convolutional neural networks (CNNs) for sentence classification using pre-trained word vectors, such as word2vec, to explore their "universal" feature extraction capabilities across diverse benchmarks. Experiments demonstrate that CNNs achieve strong performance with minimal hyperparameter tuning, highlighting their robustness and adaptability. Further improvements are obtained by fine-tuning task-specific word vectors, while a modified architecture combining static and task-specific vectors via multiple channels yields significant gains on 4 out of 7 datasets. The results underscore the effectiveness of pre-trained word vectors in capturing semantic relationships without explicit domain adaptation, offering a simple yet powerful framework for sentence-level classification tasks.
